# S - Single Responsibility (Responsabilidad Única)
Ejemplo en dominio de juegos (mini-RPG). Primero la versión que viola SRP, luego la corrección separando responsabilidades.


In [1]:
# --- Viola SRP: GameManager realiza guardado, render y lógica de juego ---
class GameManager:
    def __init__(self, player_name: str):
        # atributos de estado de juego
        self.state = 'start'
        self.player = {'name': player_name, 'hp': 100}
        self.level = 1
        self.save_path = 'save_violation.json'
    def save(self):
        import json
        with open(self.save_path, 'w') as f:
            json.dump({'player': self.player, 'level': self.level}, f)
    def load(self):
        import json
        with open(self.save_path, 'r') as f:
            data = json.load(f)
            self.player = data['player']
            self.level = data['level']
    def render(self):
        # render textual simple
        print(f"[VIOLATION] Player {self.player['name']} HP:{self.player['hp']} Lvl:{self.level}")
    def attack(self, damage: int) -> int:
        self.player['hp'] -= damage
        return self.player['hp']

gm = GameManager('Hero')
gm.render()
print('After attack hp=', gm.attack(10))
# Aquí la clase mezcla responsabilidades: guardar/recuperar, render e interacción -> difícil de testear y mantener


[VIOLATION] Player Hero HP:100 Lvl:1
After attack hp= 90


In [2]:
# --- Versión corregida: separar SaveManager y RenderManager ---
class SaveManager:
    def __init__(self, path: str):
        self.path = path
        self.last_saved = None
    def save(self, player: dict, level: int):
        import json, time
        with open(self.path, 'w') as f:
            json.dump({'player': player, 'level': level}, f)
        self.last_saved = time.time()
    def load(self) -> tuple:
        import json
        with open(self.path, 'r') as f:
            data = json.load(f)
        return data['player'], data['level']

class RenderManager:
    def __init__(self, style: str = 'text'):
        self.style = style
        self.frames_rendered = 0
    def render_player(self, player: dict, level: int):
        # render simple basado en estilo
        if self.style == 'text':
            print(f"[RENDER] Player {player['name']} HP:{player['hp']} Lvl:{level}")
        self.frames_rendered += 1
    def render_status(self):
        print(f"Frames: {self.frames_rendered}")

class GameManagerFixed:
    def __init__(self, player_name: str, save_mgr: SaveManager, render_mgr: RenderManager):
        self.state = 'start'
        self.player = {'name': player_name, 'hp': 100}
        self.level = 1
        self.save_mgr = save_mgr
        self.render_mgr = render_mgr
    def attack(self, damage: int) -> int:
        self.player['hp'] -= damage
        return self.player['hp']
    def save_game(self):
        self.save_mgr.save(self.player, self.level)
    def render(self):
        self.render_mgr.render_player(self.player, self.level)

# uso
save_mgr = SaveManager('db.json')
render_mgr = RenderManager('text')
gmf = GameManagerFixed('Hero', save_mgr, render_mgr)
gmf.render()
print('After attack hp=', gmf.attack(20))
gmf.save_game()
print('Save timestamp:', save_mgr.last_saved)


[RENDER] Player Hero HP:100 Lvl:1
After attack hp= 80
Save timestamp: 1788021994.9349701
